# QickworkspaceV2 Temperature Loop

Measure T2 echo and T1 for Q1, Q3, and Q4 at each temperature. Each temperature gets one 2x3 summary figure: T2E on the top row, T1 on the bottom row.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from time import sleep

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, update_display
from qick.asm_v2 import QickSweep1D
from tqdm.auto import tqdm

from QickworkspaceV2 import BaseExperiment
from QickworkspaceV2.config.system_cfg import config_list
from QickworkspaceV2.tools.system_tool import ExperimentConfig
from QickworkspaceV2.experiments.coherence import SpinEcho, T1
from QickworkspaceV2.core.base_analysis import BaseAnalysis
from QickworkspaceV2.tools.fitting import decaysin, expfunc

plt.ioff()

In [ ]:
# QICK connection. Run once per kernel.
BaseExperiment.connect_pyro4(
    ns_host="192.168.10.82",
    ns_port=8888,
    proxy_name="myqick",
    data_path=r"D:\Labber_Data\Jay\purcell_tmon\Rshield\temperature",
)

In [ ]:
# Measurement settings
TEMPERATURES_MK = [210]
QUBITS = ["Q1", "Q3", "Q4"]
PY_AVG = 50
STEPS = 100
RELAX_DELAY_US = 50
SLEEP_BEFORE_EACH_TEMP_MIN = 10

T2E_STOP_US = 150
T2E_RAMSEY_FREQ_MHZ = 2
T1_STOP_US = 250

CSV_DIR = Path("Rshield") / "mic"
CSV_DIR.mkdir(parents=True, exist_ok=True)

# Set False while debugging if you do not want to write Labber files.
SAVE_LABBER = True

In [ ]:
def _result_value(result, key):
    if result is None or not getattr(result, "fit_result", None):
        return None, None
    return result.fit_result.get(key, (None, None))


def _plot_result_on_ax(ax, result, simfunc, value_key, title):
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Delay time (us)")
    ax.set_ylabel("ADC")
    ax.grid(True, alpha=0.25)

    if result is None or result.x_axis is None or result.raw_iq is None:
        ax.text(0.5, 0.5, "not measured", ha="center", va="center", transform=ax.transAxes)
        return

    channel = result.metadata.get("fit_channel", "abs")
    y = BaseAnalysis._channel_data(result.raw_iq, channel)
    x = np.asarray(result.x_axis)
    ax.plot(x, y, "o-", ms=3, lw=0.9, label=channel)

    if result.fit_params is not None:
        x_fit = np.linspace(float(np.nanmin(x)), float(np.nanmax(x)), 600)
        try:
            ax.plot(x_fit, simfunc(x_fit, *result.fit_params), "r-", lw=1.6, label="fit")
        except Exception:
            pass

    val, err = _result_value(result, value_key)
    lines = []
    if val is not None:
        if err is None:
            lines.append(f"{value_key} = {val:.2f} us")
        else:
            lines.append(f"{value_key} = {val:.2f} ± {err:.2f} us")
    quality = getattr(result, "quality", None)
    if quality is not None:
        lines.append(str(getattr(quality, "value", quality)))
    if lines:
        ax.text(
            0.98, 0.96, "
".join(lines),
            ha="right", va="top", transform=ax.transAxes,
            fontsize=8, family="monospace",
            bbox=dict(facecolor="white", edgecolor="0.7", alpha=0.85),
        )
    ax.legend(fontsize=7, loc="best")


def plot_temperature_summary(results_by_qubit, temperature_mK, display_id=None):
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=False)
    fig.suptitle(f"Temperature {temperature_mK} mK", fontsize=14, fontweight="bold")

    for col, qubit in enumerate(QUBITS):
        _plot_result_on_ax(
            axes[0, col],
            results_by_qubit.get(qubit, {}).get("t2e"),
            decaysin,
            "T2e_us",
            f"{qubit} T2E",
        )
        _plot_result_on_ax(
            axes[1, col],
            results_by_qubit.get(qubit, {}).get("t1"),
            expfunc,
            "T1_us",
            f"{qubit} T1",
        )

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    if display_id is None:
        display(fig)
    else:
        display(fig, display_id=display_id, update=True)
    plt.close(fig)


def append_csv_row(filename, row):
    df = pd.DataFrame([row])
    file_exists = Path(filename).is_file()
    df.to_csv(filename, mode="a", index=False, header=not file_exists)

In [ ]:
def run_t2e(config_all, qubit):
    run_cfg = config_all.get_qubit(qubit)
    run_cfg.update([
        ("steps", STEPS),
        ("relax_delay", RELAX_DELAY_US),
        ("wait_time", QickSweep1D("waitloop", 0.0, T2E_STOP_US)),
        ("ramsey_freq", T2E_RAMSEY_FREQ_MHZ),
    ])
    expt = SpinEcho(run_cfg)
    result = expt.run(PY_AVG)
    return expt, result


def run_t1(config_all, qubit):
    run_cfg = config_all.get_qubit(qubit)
    run_cfg.update([
        ("steps", STEPS),
        ("relax_delay", RELAX_DELAY_US),
        ("wait_time", QickSweep1D("waitloop", 0.0, T1_STOP_US)),
    ])
    expt = T1(run_cfg)
    result = expt.run(PY_AVG)
    return expt, result

In [ ]:
def _fmt_us(value, err=None):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "nan"
    if err is None or (isinstance(err, float) and np.isnan(err)):
        return f"{value:.2f} us"
    return f"{value:.2f}±{err:.2f} us"


all_results = {}

for temperature in TEMPERATURES_MK:
    if SLEEP_BEFORE_EACH_TEMP_MIN > 0:
        print(f"Waiting {SLEEP_BEFORE_EACH_TEMP_MIN} min before {temperature} mK measurement...")
        sleep(SLEEP_BEFORE_EACH_TEMP_MIN * 60)

    config_all = ExperimentConfig(config_list)
    results_by_qubit = {qubit: {} for qubit in QUBITS}
    all_results[temperature] = results_by_qubit
    csv_file = CSV_DIR / f"qubit_results_{temperature}mK.csv"
    summary_display_id = f"temperature-summary-{temperature}mK"

    # Show an empty 2x3 figure immediately, then update it after every completed measurement.
    plot_temperature_summary(results_by_qubit, temperature, display_id=summary_display_id)

    pbar = tqdm(QUBITS, desc=f"{temperature} mK")
    for qubit in pbar:
        pbar.set_description(f"{temperature} mK | {qubit} T2E")
        t2e_expt, t2e_result = run_t2e(config_all, qubit)
        results_by_qubit[qubit]["t2e"] = t2e_result
        if SAVE_LABBER:
            t2e_expt.saveLabber(qb_idx=qubit, config_all=config_all, title=f"{temperature}mK")
        plot_temperature_summary(results_by_qubit, temperature, display_id=summary_display_id)
        plt.close("all")

        pbar.set_description(f"{temperature} mK | {qubit} T1")
        t1_expt, t1_result = run_t1(config_all, qubit)
        results_by_qubit[qubit]["t1"] = t1_result
        if SAVE_LABBER:
            t1_expt.saveLabber(qb_idx=qubit, config_all=config_all, title=f"{temperature}mK")
        plot_temperature_summary(results_by_qubit, temperature, display_id=summary_display_id)
        plt.close("all")

        t2e_val, t2e_err = _result_value(t2e_result, "T2e_us")
        t1_val, t1_err = _result_value(t1_result, "T1_us")
        row = {
            "temperature_mK": temperature,
            "qubit": qubit,
            "T2e_us": t2e_val,
            "T2e_err_us": t2e_err,
            "T1_us": t1_val,
            "T1_err_us": t1_err,
            "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
        append_csv_row(csv_file, row)
        tqdm.write(
            f"{temperature} mK | {qubit} | "
            f"T2E={_fmt_us(t2e_val, t2e_err)} | "
            f"T1={_fmt_us(t1_val, t1_err)}"
        )

    plot_temperature_summary(results_by_qubit, temperature, display_id=summary_display_id)
    plt.ion()
    tqdm.write(f"{temperature} mK measurement block finished.")

In [ ]:
# Optional: re-plot any temperature summary after the loop.
# plot_temperature_summary(all_results[210], 210)